# Solving an Inverse Projectile Problem with a Genetic Algorithm

## Objective

In this assignment you will implement a **Genetic Algorithm (GA)** with **binary encoding** to solve an inverse projectile problem. The goal is to determine the launch parameters of a **cannon-like launcher** so that a projectile reaches a desired horizontal distance.

You will work with:

- binary encoding of continuous variables  
- chromosome decoding  
- fitness function design  
- experimentation with GA parameters  
- interactive visualization using `ipywidgets`

Cells labeled **Student Task** require you to implement or modify code.  
Cells without this label contain code that has already been provided.

---

# Problem Description

### Rescue Capsule Launch System

In a mountainous region, a launcher has been installed to deliver **capsules containing medical supplies** to an isolated community.

The system uses a launching system that accelerates a capsule of mass $m$ along a launch tube.

The launcher applies a **constant force $F$** on the capsule while it travels inside a tube of length $L$. When the capsule exits the tube, it leaves the launcher with an initial velocity $v_0$ forming an angle $\theta$ with respect to the horizontal.

After leaving the launcher, the capsule follows a **projectile trajectory** under the influence of gravity alone (air resistance is neglected).

---

## Projectile Motion Model

The initial velocity generated by the launcher is determined from the **work–energy principle**:

$$
v_0 = \sqrt{\frac{2FL}{m}}
$$

If the projectile is fired at an angle $\theta$ relative to the horizontal, the **horizontal landing range** of the projectile is

$$
R_l = \frac{v_0^2 \sin(2\theta)}{g}
$$

where

$$
g = 9.81\ \text{m/s}^2
$$

---

## Maximum Height of the Projectile

The maximum height reached by the projectile during its trajectory is given by

$$
h_{max} = \frac{v_0^2 \sin^2(\theta)}{2g}
$$

This value represents the highest vertical position reached before the projectile begins to descend.

---

## Inverse Problem

Instead of computing the range from the launch parameters, you must solve the **inverse problem**:

> Given a target horizontal distance (goal range) $R_g$ and a projectile mass $m$, use a Genetic Algorithm to determine the values of $L$, $\theta$, and $F$ that produce a landing range $R_l$ as close as possible to $R_g$.

---

# Parameter and Objective Constraints 

The parameters must satisfy the following ranges:

$$
L \in [1,\ 20]\ \text{m}
$$

$$
F \in [500,\ 5000]\ \text{N}
$$

$$
\theta \in [10^\circ,\ 80^\circ]
$$

The target distance must satisfy

$$
R_g \in [100,\ 2000]
$$

The projectile mass will be fixed at

$$
m = 10\ \text{kg}
$$

---

# Notebook Structure and Tasks

The following sections correspond to the cells in this notebook.

When a section is marked **Student Task**, you must implement or modify the code in the corresponding cell.

---

## 1. Constants Definition

**Student Task**

Define the constants required for the problem, such as:

- gravitational acceleration $g$
- projectile mass $m$
- maximum plotting range
- parameter bounds

---

## 2. Chromosome Decoding Functions

**Student Task**

Implement the functions that decode a binary chromosome into the real-valued parameters:

- $L$
- $\theta$
- $F$

Use the decoding formula:

$$
x = x_{min} + \frac{integer}{2^n - 1}(x_{max} - x_{min})
$$

Your decoding function should:

- extract the binary segments corresponding to each variable
- convert them to integers
- map them to their corresponding real intervals

---

## 3. Distance Function

**Student Task**

Implement the function that computes the **landing range** $R_l$ of the projectile using the physical model described earlier.

Your function should compute:

- initial velocity $v_0$
- landing range $R_l$

---

## 4. Goal Range Drawing

The notebook includes a function that draws:

- the launcher position
- the target location $R_g$
- the projectile trajectory

This code has already been provided.

---

## 5. Fitness Function

**Student Task**

Define the **fitness function** that evaluates how close the projectile landing range is to the target range.

Possible definitions include:

$$
error = |R_g - R_l|
$$

or

$$
error = (R_g - R_l)^2
$$

The goal of the Genetic Algorithm is to **minimize this error**.

---

## 6. GA Definition

**Student Task**

Define the main components of the Genetic Algorithm:

- fitness function registration
- gene generation
- individual representation
- population generation

These definitions determine how candidate solutions are represented and evaluated.

---

## 7. Single Experiment Runner

This section contains code that runs **a single Genetic Algorithm experiment** using a fixed set of parameters.

The code has already been provided.

---

## 8. Multiple Experiments Runner

This section contains code that runs **multiple GA experiments automatically** to analyze algorithm performance.

The code has already been provided.

---

## 9. Target Distance Selection Interface

The notebook includes an **interactive interface using `ipywidgets`** that allows the user to select the goal distance $R_g$.

The interface also visualizes the resulting projectile trajectory.

This code has already been provided.

---

## 10. Fixed Parameters Experiment

**Student Task**

Run an experiment with a fixed set of GA parameters and analyze the result.

Observe:

- convergence behavior
- final solution quality

---

## 11. Mutation Probability Experiments

**Student Task**

Run experiments varying the **mutation probability** and analyze how it affects:

- convergence speed
- stability of solutions
- final accuracy

---

## 12. Crossover Probability Experiments

**Student Task**

Run experiments varying the **crossover probability** and analyze its impact on the GA performance.

---

## 13. Population Size Experiments

**Student Task**

Run experiments varying the **population size**.

Discuss how population size affects:

- exploration of the search space
- convergence speed
- solution quality

---

## 14. Discussion and Conclusions

**Student Task**

Summarize your observations from the experiments.

Discuss:

- the best solutions found
- the difference between $R_g$ and $R_l$
- the influence of GA parameters
- the stability of the algorithm across runs

Note that **multiple combinations of parameters may produce similar projectile ranges**.  
Discuss how this affects the behavior of the Genetic Algorithm.

---

# Deliverables

Submit this **Jupyter Notebook** with all **Student Task sections completed**, including:

- implemented functions
- experimental results
- discussion and conclusions



In [ ]:
import sys
import random
import math
import numpy as np
import matplotlib.pyplot as plt
import math
import ipywidgets as widgets
from IPython.display import display, clear_output

from deap import base, creator, tools, algorithms

sys.path.append("..")
from common.homtransf import *
from common.robot_2D import *

### 1 Constants definition <span style="color:blue"><b>(Student task)</b></span>
Define the following constants from the problem statement.


In [ ]:
# TODO: Define the parameters for the optimization problem
MIN_LENGHT = 
MAX_LENGHT = 
N_BITS_LENGTH = 

MIN_FORCE = 
MAX_FORCE = 
N_BITS_FORCE = 

MIN_THETA =  # Must be in radians
MAX_THETA =  # Must be in radians
N_BITS_THETA = 

TOTAL_BITS = N_BITS_LENGTH + N_BITS_FORCE + N_BITS_THETA

MIN_R = 
MAX_R = 

MASS = 
G = 9.81 #Gravity acceleration in m/s^2

<div style="
    height:500px;
    width:100%;
    border:2px solid lightgray;
    padding:10px;
    padding:15px;
    border-radius:8px;
    box-sizing:border-box;
">

### 2. Chromosome decoding functions <span style="color:blue"><b>(Student task)</b></span>
Insert an image representing the cromossoma decoding functions and implement the decoding functions, then erase this coment. 
<div style="text-align:center;">
    <img src="../images/step.png" style="height:380px; width:auto;">
</div>
</div>

In [ ]:
def bits_decoding(bits, x_min, x_max):
 

    n = len(bits)

    # TODO: Convert the binary list to an integer
    integer = 

    # TODO:Linearly scale the integer to the interval [x_min, x_max]
    value = 

    return value

def bits_to_angle(bits):

    theta = # TODO:Use the bits_decoding function to convert the binary list to an angle in radians

    return theta

def bits_to_length(bits):

    length =  # TODO:Use the bits_decoding function to convert the binary list to a length in meters  

    return length

def bits_to_force(bits):

    force =  # TODO:Use the bits_decoding function to convert the binary list to a force in Newtons

    return force


def decode_individual(individual):

    length_bits =  # TODO: Extract the first N_BITS_LENGTH bits for length
    force_bits =  # TODO: Extract the next N_BITS_FORCE bits for force
    theta_bits =  # TODO: Extract the remaining bits for theta

    length =  # TODO: Use the bits_to_length function to convert the binary list to a length in meters
    force = # TODO: Use the bits_to_force function to convert the binary list to a force in Newtons
    theta = # TODO: Use the bits_to_angle function to convert the binary list to an angle in radians

    return length, force, theta

<div style="
    height:500px;
    width:100%;
    border:2px solid lightgray;
    padding:10px;
    padding:15px;
    border-radius:8px;
    box-sizing:border-box;
">

### 3. Distance function <span style="color:blue"><b>(Student task)</b></span>
Insert an image representing the distance function and implement it, then erase this coment. 

<div style="text-align:center;">
    <img src="../images/step.png" style="height:400px; width:auto;">
</div>
</div>


In [ ]:
def distance(r_g, r_l):
 
    distance =  # TODO: Calculate the distance between r_l and r_g

    return distance

### 4. Goal Range Drawing

In [ ]:
def _setup_range_plot(r_g, y_max, title):
    """
    Creates the common plot structure:
    - launch point at (0, 0)
    - target at (r_g, 0)
    - fixed x-axis up to MAX_R
    """
    y_max = y_max * 0.1

    plt.figure(figsize=(8, 5))

    # Launch point
    plt.scatter(0, 0, s=120)
    plt.text(0, -0.05 * y_max, "Launch (0,0)", ha="center")

    # Target
    plt.scatter(r_g, 0, s=150)
    plt.text(r_g, 0, "X", fontsize=22, ha="center", va="center")
    plt.text(r_g, -0.1 * y_max, f"Target ({r_g:.2f}, 0)", ha="center")

    # Axes
    plt.xlim(0, MAX_R)
    plt.ylim(0, y_max)

    plt.xlabel("x (distance)")
    plt.ylabel("y (height)")
    plt.title(title)

    plt.axhline(0)
    plt.axvline(0)
    plt.grid(True)


def draw_target(R, y_max):
    """
    Draws only the launch point and the target.
    """
    _setup_range_plot(R, y_max, "Launch Point and Target")
    plt.show()


def draw_solution(r_g, r_landing, f, theta, l, m=MASS, g=G, n_points=300):
    """
    Draws the target at (r_g, 0) and the projectile trajectory
    produced by parameters f, theta, and l, ending at (r_landing, 0).

    Parameters
    ----------
    r_g : float
        Target horizontal distance.
    r_landing : float
        Landing distance reached by the solution.
    f : float
        Applied force.
    theta : float
        Launch angle in radians.
    l : float
        Launch tube / barrel length.
    m : float
        Projectile mass.
    g : float
        Gravity.
    n_points : int
        Number of points used to draw the parabola.
    """

    # Initial velocity from work-energy relation
    v0 = math.sqrt((2 * f * l) / m)

    # Time of flight assuming launch and landing at same height
    t_flight = (2 * v0 * math.sin(theta)) / g

    # Trajectory
    t = np.linspace(0, t_flight, n_points)
    x = v0 * math.cos(theta) * t
    y = v0 * math.sin(theta) * t - 0.5 * g * t**2

    # Theoretical maximum height for scaling
    y_peak = (v0**2 * math.sin(theta)**2) / (2 * g)

    # Use the larger of:
    # - the trajectory peak
    # - a fraction of MAX_R so small trajectories still look good
    y_max_for_plot = max(y_peak * 1.25, MAX_R * 0.25)

    _setup_range_plot(r_g, y_max_for_plot / 0.1, "Projectile Solution")

    # Parabolic trajectory
    plt.plot(x, y, linewidth=2, label="Trajectory")

    # Landing point
    plt.scatter(r_landing, 0, s=100)
    plt.text(r_landing, 0.08 * (y_max_for_plot), f"Landing ({r_landing:.2f}, 0)", ha="center")

    # Optional visual aid: vertical line at landing point
    plt.axvline(r_landing, linestyle="--", alpha=0.7)

    # Show solution parameters
    info = (
        f"F = {f:.2f} N\n"
        f"Theta = {theta:.3f} rad\n"
        f"L = {l:.2f} m\n"
        f"R_goal = {r_g:.2f} m\n"
        f"R_landing = {r_landing:.2f} m\n"
        f"Error = {abs(r_g - r_landing):.2f} m"
    )

    plt.text(
        MAX_R * 0.68,
        y_max_for_plot * 0.95,
        info,
        va="top",
        bbox=dict(boxstyle="round", alpha=0.15)
    )

    plt.legend()
    plt.show()

### 5. Define the fitness function <span style="color:blue"><b>(Student task)</b></span>
- Define the function `compute_r_landing(length, force, theta, m=MASS, g=G)` that computes $r_l$ given $lenght$ (L), $force$ (F) and $\theta$
- Using the decoding function `decode_individual(individual)` and `compute_r_landing(length, force, theta, m=MASS, g=G)` implement fitness function.

In [ ]:
# =========================================================
# 1. FITNESS FUNCTION
# =========================================================

def compute_r_landing(length, force, theta, m=MASS, g=G):

    v0 = # TODO: Compute the initial velocity using the work-energy relation
    r_l = # TODO: Compute the landing distance using the projectile motion equations

    return r_l

def eval_function(individual):
    global r_g

    # TODO: Decode the individual's binary representation into parameters
    length, force, theta = 

    # TODO: compute r_l with compute_r_landing
    r_l = 

    # TODO: compute the distance between r_l and r_g with function "distance"
    dist = 

    return (dist,)

<div style="
    height:500px;
    width:100%;
    border:2px solid lightgray;
    padding:10px;
    padding:15px;
    border-radius:8px;
    box-sizing:border-box;
">

### 6. GA definition: fitness, gene, individual, population <span style="color:blue"><b>(Student task)</b></span>

<div style="text-align:center;">
    <img src="../images/GA_Toolbox.png" style="height:400px; width:auto;">
</div>
</div>

In [ ]:
# =========================================================
# 2. CREATE DEAP CLASSES (Only once per Python session)
# =========================================================
# DEAP dynamically creates classes for individuals and fitness.
# We check first to avoid errors if the notebook cell is run multiple times.

if not hasattr(creator, "fitness"):
    creator.create("fitness", base.Fitness, weights= ) # TODO: Select the appropriate weights for the optimization problem (minimization or maximization)

if not hasattr(creator, "individual_container"):
    creator.create("individual_container", list, fitness=creator.fitness)

# =========================================================
# 3. TOOLBOX CONSTRUCTION
# =========================================================
def build_toolbox(indpb=0.05, tournsize=3):
    """
    Builds and configures the DEAP toolbox.

    Parameters
    ----------

    indpb : float
        Probability of mutating each gene.

    tournsize : int
        Tournament size used for selection.

    Returns
    -------
    toolbox : deap.base.Toolbox
        Configured toolbox containing all operators.
    """

    toolbox = base.Toolbox()

    # Each gene is either 0 or 1
    toolbox.register("my_gene", random.randint, 0, 1)

    # An individual (or chromosome) is a list of binary genes
    toolbox.register(
        "individual",
        tools.initRepeat,
        creator.individual_container,
        toolbox.my_gene,
        TOTAL_BITS
    )

    # Population = list of individuals
    toolbox.register(
        "population",
        tools.initRepeat,
        list,
        toolbox.individual
    )

    # Register evaluation function
    toolbox.register("evaluate", eval_function)

    # Genetic operators
    toolbox.register("mate", tools.cxTwoPoint)

    # Bit-flip mutation
    toolbox.register("mutate", tools.mutFlipBit, indpb=indpb)

    # Tournament selection
    toolbox.register("select", tools.selTournament, tournsize=tournsize)

    return toolbox

<div style="
    height:500px;
    width:100%;
    border:2px solid lightgray;
    padding:10px;
    padding:15px;
    border-radius:8px;
    box-sizing:border-box;
">

### 7 Single-experiment runner

<div style="text-align:center;">
    <img src="../images/GA_parameters.png" style="height:400px; width:auto;">
</div>
</div>

In [ ]:
# =========================================================
# 4. RUN A SINGLE EXPERIMENT
# =========================================================
def run_experiment(
    pop_size=50,
    cxpb=0.5,
    mutpb=0.2,
    n_gen=40,
    indpb=0.05,
    seed=0
):
    """
    Runs a complete genetic algorithm experiment.

    Parameters
    ----------
    pop_size : int
        Number of individuals in the population.

    cxpb : float
        Probability of applying crossover.

    mutpb : float
        Probability of mutating an individual.

    n_gen : int
        Number of generations.

    ind_size : int
        Length of each individual's genome.

    indpb : float
        Probability of mutating each gene.

    seed : int
        Random seed for reproducibility.

    Returns
    -------
    dict
        Dictionary containing statistics and the best solution found.
    """

    random.seed(seed)
    np.random.seed(seed)

    toolbox = build_toolbox(indpb=indpb)

    # Create initial population
    population = toolbox.population(n=pop_size)

    # Statistics to track during evolution
    stats = tools.Statistics(lambda ind: ind.fitness.values[0])
    stats.register("avg", np.mean)
    stats.register("max", np.max)
    stats.register("min", np.min)

    # Hall of Fame keeps the best solution
    hof = tools.HallOfFame(1)

    # Run the evolutionary algorithm (In this case Genetic Algorithm)
    population, logbook = algorithms.eaSimple(
        population,
        toolbox,
        cxpb=cxpb,
        mutpb=mutpb,
        ngen=n_gen,
        stats=stats,
        halloffame=hof,
        verbose=False
    )

    # Extract statistics from the logbook
    generations = logbook.select("gen")
    avg_fitness = logbook.select("avg")
    max_fitness = logbook.select("max")
    min_fitness = logbook.select("min")

    return {
        "generations": generations,
        "avg": avg_fitness,
        "max": max_fitness,
        "min": min_fitness,
        "best_individual": hof[0],
        "best_fitness": hof[0].fitness.values[0]
    }



<div style="
    height:500px;
    width:100%;
    border:2px solid lightgray;
    padding:10px;
    padding:15px;
    border-radius:8px;
    box-sizing:border-box;
">

### 8 Multiple-experiments runner

<div style="text-align:center;">
    <img src="../images/GA_list_of_parameters.png" style="height:400px; width:auto;">
</div>
</div>

In [ ]:
# =========================================================
# 5. COMPARE MULTIPLE EXPERIMENTS
# =========================================================
def compare_experiments(configs, n_gen=40, indpb=0.05, seed=0):
    """
    Runs multiple GA experiments with different parameters and plots results.

    Parameters
    ----------
    configs : list of dict
        List of parameter configurations to test.

    n_gen : int
        Number of generations.

    ind_size : int
        Genome length.

    indpb : float
        Gene mutation probability.

    seed : int
        Random seed.
    """

    plt.figure(figsize=(10, 6))

    all_results = []

    for config in configs:

        result = run_experiment(
            pop_size=config["pop_size"],
            cxpb=config["cxpb"],
            mutpb=config["mutpb"],
            n_gen=n_gen,
            indpb=indpb,
            seed=seed
        )

        all_results.append(result)

        label = (
            f'pop={config["pop_size"]}, '
            f'cxpb={config["cxpb"]}, '
            f'mutpb={config["mutpb"]}'
        )

        plt.plot(result["generations"], result["min"], label=label)

    plt.xlabel("Generation")
    plt.ylabel("Best Fitness")
    plt.title("Genetic Algorithm Convergence Comparison")
    plt.legend()
    plt.grid(True)
    plt.show()

    # Draws the best solution of each experiment
    for i, result in enumerate(all_results):
        best_individual = result["best_individual"]
        best_fitness = result["best_fitness"]

        length, force, theta = decode_individual(best_individual)

        r_landing = compute_r_landing(length, force, theta)
        
        draw_solution(r_g, r_landing, force, theta, length)

        print("Best individual:", best_individual)
        print("Best fitness:", best_fitness)

### 9. $R_g$ selection interface

In [ ]:
r_g = 1000
# Output area
out = widgets.Output()

# Sliders
g_slider = widgets.FloatSlider(
    value=r_g, min=MIN_R, max=MAX_R, step=(MAX_R - MIN_R)/200,
    description="r_g", continuous_update=False
)

def update_plot(r_g_f):
    global r_g
    with out:
        clear_output(wait=True)

        r_g = r_g_f

        draw_target(r_g, y_max=MAX_R)


controls = widgets.interactive_output(
    update_plot,
    {
        "r_g_f": g_slider,
    }
)

ui = widgets.VBox([
    widgets.HBox([g_slider]),
    out
])

display(ui)

### 10. Fixed parameters experiment <span style="color:blue"><b>(Student task)</b></span>

In [ ]:
# =========================================================
# 6. EXAMPLE USAGE
# =========================================================

# Run a baseline experiment
base_result = run_experiment(
    pop_size= , # TODO: Select the population size
    cxpb= , # TODO: Select the crossover probability
    mutpb= , # TODO: Select the mutation probability
    n_gen= , # TODO: Select the number of generations
    indpb= , # TODO: Select the gene mutation probability
    seed=  # TODO: Select the random seed for reproducibility
)

best_individual = base_result["best_individual"] # Get the best individual from the results
best_fitness = base_result["best_fitness"] # Get the best fitness value from the results

length, force, theta = decode_individual(best_individual) # Decode the best individual's binary representation into length, force, and theta
r_landing = compute_r_landing(length, force, theta) # Compute the landing distance of the projectile using the decoded parameters
draw_solution(r_g, r_landing, force, theta, length) # Draw the solution on the plot

print("Best individual:", best_individual)

### 11. Mutation probability experiments <span style="color:blue"><b>(Student task)</b></span>

In [ ]:
# ---------------------------------------------------------
# Experiment 1: Vary mutation probability
# ---------------------------------------------------------

# TODO: Fill in the configs_mutation list with different mutation probabilities while keeping other parameters constant.
configs_mutation = [
    {"pop_size": , "cxpb": , "mutpb": },
    {"pop_size": , "cxpb": , "mutpb": },
    {"pop_size": , "cxpb": , "mutpb": },
]

compare_experiments(configs_mutation, n_gen=40, indpb=0.05, seed=42)

### 12.  Crossover probability experiments <span style="color:blue"><b>(Student task)</b></span>

In [ ]:
# ---------------------------------------------------------
# Experiment 2: Vary crossover probability
# ---------------------------------------------------------

# TODO: Fill in the configs_crossover list with different crossover probabilities while keeping other parameters constant.
configs_crossover = [
    {"pop_size": , "cxpb": , "mutpb": },
    {"pop_size": , "cxpb": , "mutpb": },
    {"pop_size": , "cxpb": , "mutpb": },
]

compare_experiments(configs_crossover, n_gen=40, indpb=0.05, seed=42)

### 13. Population size experiments <span style="color:blue"><b>(Student task)</b></span>

In [ ]:
# ---------------------------------------------------------
# Experiment 3: Vary population size
# ---------------------------------------------------------

# TODO: Fill in the configs_population list with different population sizes while keeping other parameters constant.
configs_population = [
    {"pop_size": , "cxpb": , "mutpb": },
    {"pop_size": , "cxpb": , "mutpb": },
    {"pop_size": , "cxpb": , "mutpb": },
]

compare_experiments(configs_population, n_gen=40, indpb=0.05, seed=42)

### 14 Discussion and conclusions <span style="color:blue"><b>(Student task)</b></span>
Write here your discussion and conclusions here, then erase this line.
